# YOLOv8m — fair RDD2022 comparison

COCO-pretrained `yolov8m.pt`, 640 px, SGD lr 0.01, batch 16 (`nbs=16`).
Ultralytics' extra augmentations (mosaic, mixup, translate, HSV hue/sat, Albumentations) are **switched off** so YOLO trains under the same augmentation policy as SSD and Faster R-CNN.
Model selection uses the shared evaluator (callback after each epoch), not Ultralytics' fitness.

Saved to Drive under `MyDrive/RDD_fair/results/yolov8m/seed0/`
(`checkpoints/best.pt` = selected epoch, `ultralytics/weights/last.pt` = resume point, `logs/train_log.csv`).

**If Colab disconnects:** reconnect and *Run all* again. A finished run is skipped; an unfinished run resumes from its last saved epoch.

In [ ]:
# 0) Ultralytics silently adds Blur/CLAHE/ToGray when albumentations is installed -> remove it
!pip uninstall -y albumentations
import importlib.util
assert importlib.util.find_spec('albumentations') is None, 'Restart the runtime (Runtime > Restart session) and run again'

In [ ]:
# 1) Mount Drive and import the shared code (upload the folder rdd_fair/ to MyDrive/RDD_fair/code/)
from google.colab import drive
drive.mount('/content/drive')
import sys, os
CODE_DIR = '/content/drive/MyDrive/RDD_fair/code'
sys.path.insert(0, CODE_DIR)
!nvidia-smi

In [ ]:
# 2) Same library versions in every session, same GPU type, same dataset
from rdd_fair import config as C, utils as U, prepare_data as P
U.pin_or_install_packages()
C.ensure_dirs()
U.check_gpu_consistency()
data_yaml = P.prepare_rdd()      # first time: creates the fixed split; later: rebuilds it and verifies hashes

In [ ]:
# 3) Train
from rdd_fair import yolo_train as Y
from rdd_fair.yolo_train import train_args
print({k: v for k, v in train_args(0, data_yaml).items() if k not in ('project',)})
for seed in C.SEEDS:
    Y.train(seed, data_yaml)

In [ ]:
# 4) Training curves (fair_eval on val each epoch)
import pandas as pd, matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
for seed in C.SEEDS:
    p = os.path.join(C.run_dir('yolov8m', seed, 'logs'), 'train_log.csv')
    if os.path.exists(p):
        d = pd.read_csv(p).drop_duplicates('epoch', keep='last')
        ax.plot(d.epoch, d.val_mAP50_95, label=f'seed {seed} mAP50-95')
        ax.plot(d.epoch, d.val_mAP50, '--', label=f'seed {seed} mAP50')
ax.legend(fontsize=8); ax.set_title('val mAP (fair_eval)'); plt.show()
for seed in C.SEEDS:
    print(seed, U.load_json(os.path.join(C.run_dir('yolov8m', seed, 'logs'), 'selection.json')))

## Optional: peak GPU memory during training (Table 5)

Not needed for the main results. Runs one short Ultralytics training pass (2% of the training images, one epoch) with the same batch size, input size and augmentation settings as the real run, in a temporary folder, and reports the peak memory allocated by PyTorch.

In [ ]:
import glob, yaml, torch
from ultralytics import YOLO
args = yaml.safe_load(open(glob.glob(os.path.join(C.run_dir('yolov8m', 0), '**', 'args.yaml'), recursive=True)[0]))
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
YOLO('yolov8m.pt').train(data=args['data'], imgsz=640, batch=16, epochs=1, fraction=0.02,
     mosaic=0, mixup=0, translate=0, degrees=0, hsv_h=0, hsv_s=0, hsv_v=0.2, scale=0.1, fliplr=0.5,
     project='/content/mem_test', name='yolo', val=False, plots=False)
print('Training peak GPU memory (MB):', round(torch.cuda.max_memory_allocated() / 1024**2))